# 🏛️ BharatCRS V7 - Transformers Multi-Task Training
This notebook trains the **6-domain, 20-issue** civic classifier using the Muril (google/muril-base-cased) backbone.

### 📋 Instructions:
1. **Upload**: Upload `bharatcrs_v7_clean.csv` and `label_maps_v7.json` (generated by `prepare_v7.py`) to the file sidebar.
2. **Train**: Run all cells. 
3. **Result**: Your `bharatcrs_v7.onnx` will be ready for download in 15-30 minutes.

In [ ]:
!pip install transformers datasets torch onnx onnxruntime pandas numpy

In [ ]:
import torch
import pandas as pd
import numpy as np
import json
import os
from transformers import AutoTokenizer, AutoModel, AutoConfig
from torch import nn
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

# --- CONFIG ---
MODEL_NAME = 'google/muril-base-cased'
MAX_LEN = 256
BATCH_SIZE = 8
EPOCHS = 15
LR = 1e-5
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {DEVICE}")

# --- LOAD LABELS ---
with open('label_maps_v7.json', 'r') as f:
    LABELS = json.load(f)
    DOMAIN_LABELS = LABELS['primary_domain_labels']
    ISSUE_LABELS = LABELS['issue_type_labels']

print(f"Domains: {len(DOMAIN_LABELS)} | Issues: {len(ISSUE_LABELS)}")

In [ ]:
class CivicDataset(Dataset):
    def __init__(self, csv_file, tokenizer, max_len):
        self.df = pd.read_csv(csv_file)
        self.tokenizer = tokenizer
        self.max_len = max_len
        
    def __len__(self):
        return len(self.df)
        
    def __getitem__(self, item):
        row = self.df.iloc[item]
        text = str(row['raw_text'])
        
        inputs = self.tokenizer(
            text, 
            max_length=self.max_len, 
            padding='max_length', 
            truncation=True, 
            return_tensors='pt'
        )
        
        return {
            'input_ids': inputs['input_ids'].flatten(),
            'attention_mask': inputs['attention_mask'].flatten(),
            'domain': torch.tensor(DOMAIN_LABELS.index(row['primary_domain'])),
            'issue': torch.tensor(ISSUE_LABELS.index(row['issue_type'])),
            'severity': torch.tensor(float(row['severity_level'] - 1) / 9.0, dtype=torch.float),
            'safety': torch.tensor(1.0 if row['public_safety_flag'] else 0.0, dtype=torch.float),
            'vuln': torch.tensor(1.0 if row.get('vulnerable_population_flag') else 0.0, dtype=torch.float)
        }

In [ ]:
class MultiTaskCivicClassifier(nn.Module):
    def __init__(self, n_domains, n_issues):
        super().__init__()
        self.bert = AutoModel.from_pretrained(MODEL_NAME)
        hidden_size = self.bert.config.hidden_size
        
        self.domain_head = nn.Linear(hidden_size, n_domains)
        self.issue_head = nn.Linear(hidden_size, n_issues)
        self.severity_head = nn.Sequential(nn.Linear(hidden_size, 1), nn.Sigmoid())
        self.safety_head = nn.Linear(hidden_size, 1)
        self.vuln_head = nn.Linear(hidden_size, 1)
        
    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        pooled = outputs.pooler_output
        
        return (
            self.domain_head(pooled), 
            self.issue_head(pooled), 
            self.severity_head(pooled).squeeze(-1), 
            self.safety_head(pooled).squeeze(-1), 
            self.vuln_head(pooled).squeeze(-1)
        )

In [ ]:
def train_epoch(model, data_loader, optimizer, device):
    model.train()
    losses = []
    
    ce_loss = nn.CrossEntropyLoss()
    mse_loss = nn.MSELoss()
    bce_loss = nn.BCEWithLogitsLoss()
    
    for data in tqdm(data_loader):
        input_ids = data['input_ids'].to(device)
        attention_mask = data['attention_mask'].to(device)
        d_target = data['domain'].to(device)
        i_target = data['issue'].to(device)
        sev_target = data['severity'].to(device)
        saf_target = data['safety'].to(device)
        v_target = data['vuln'].to(device)
        
        d_logits, i_logits, sev_pred, saf_logit, v_logit = model(input_ids, attention_mask)
        
        l_domain = ce_loss(d_logits, d_target)
        l_issue = ce_loss(i_logits, i_target)
        l_sev = mse_loss(sev_pred, sev_target)
        l_saf = bce_loss(saf_logit, saf_target)
        l_vuln = bce_loss(v_logit, v_target)
        
        # Weights: Emphasize Domain and Issue Identification
        total_loss = (2.0 * l_domain) + (2.0 * l_issue) + (0.5 * l_sev) + (1.0 * l_saf) + (1.0 * l_vuln)
        
        optimizer.zero_grad()
        total_loss.backward()
        optimizer.step()
        losses.append(total_loss.item())
        
    return np.mean(losses)

# --- INIT ---
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
train_ds = CivicDataset('bharatcrs_v7_clean.csv', tokenizer, MAX_LEN)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)

model = MultiTaskCivicClassifier(len(DOMAIN_LABELS), len(ISSUE_LABELS)).to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR)

for epoch in range(EPOCHS):
    loss = train_epoch(model, train_loader, optimizer, DEVICE)
    print(f"Epoch {epoch+1}/{EPOCHS} | Loss: {loss:.4f}")

In [ ]:
# --- EXPORT TO ONNX ---
model.eval()
dummy_input = torch.ones(1, MAX_LEN, dtype=torch.long).to(DEVICE)
dummy_mask = torch.ones(1, MAX_LEN, dtype=torch.long).to(DEVICE)

torch.onnx.export(
    model, 
    (dummy_input, dummy_mask), 
    'bharatcrs_v7.onnx', 
    input_names=['input_ids', 'attention_mask'],
    output_names=['domain_logits', 'issue_logits', 'severity_pred', 'safety_logit', 'vulnerable_logit'],
    dynamic_axes={'input_ids': {0: 'batch_size'}, 'attention_mask': {0: 'batch_size'}},
    opset_version=12
)
print("Deployment Model Exported: bharatcrs_v7.onnx")